# Cold Start SFT Data Preprocessing — Google Colab
**Preprocesses Sky-T1 and OpenThoughts CoT datasets and pushes to Hugging Face Dataset Hub**

This notebook runs in Google Colab. It cleans raw CoT datasets, standardizes thinking `<think>...</think>` and answer `<answer>...</answer>` tags, and uploads the formatted dataset to Hugging Face as `abhinav0231/reasoning-cold-start-sft-data`.

## Cell 1 — Install Dependencies & Authentication

In [ ]:
# ==============================================================================
# Robust Authentication (Hugging Face & Weights & Biases)
# ==============================================================================
import os
from huggingface_hub import login, HfFolder

# Hugging Face Authentication
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = HfFolder.get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    try:
        login(token=HF_TOKEN)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

# WandB Authentication
try:
    import wandb
    WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "")
    if WANDB_TOKEN and WANDB_TOKEN != "YOUR_WANDB_KEY_HERE":
        wandb.login(key=WANDB_TOKEN, relogin=True)
        os.environ["WANDB_API_KEY"] = WANDB_TOKEN
        print("✅ Authenticated with Weights & Biases")
    else:
        print("ℹ️ WANDB_API_KEY not found. WandB tracking will operate in offline/disabled mode.")
        os.environ["WANDB_DISABLED"] = "true"
except ImportError:
    print("ℹ️ WandB module not installed. Operating without WandB tracking.")
    os.environ["WANDB_DISABLED"] = "true"


## Cell 2 — Configuration & Global Parameters

In [ ]:
# ==============================================================================
# Cell 2 — Global Hyperparameters & Dataset Configuration
# ==============================================================================

# Hugging Face user credentials and target repository for the preprocessed dataset
HF_USERNAME = "abhinav0231"
OUTPUT_DATASET_REPO = f"{HF_USERNAME}/reasoning-cold-start-sft-data"

# Sampling Limits for Cold-Start SFT Dataset (Total Target: 4,000 samples)
# - Sky-T1 provides diverse reasoning across math, coding, and general logic (2,500 samples)
# - OpenThoughts provides complex, high-difficulty mathematical CoT reasoning (1,500 samples)
MAX_SAMPLES_SKYT1       = 2500    
MAX_SAMPLES_OPENTHOUGHT = 1500    
SEED                    = 42      # Reproducibility seed for dataset shuffling

# Standardized System Prompt to enforce explicit reasoning structure in the model
# Instructs the model to isolate step-by-step reasoning inside <think>...</think>
# and output the final verified answer inside <answer>...</answer> tags.
SYSTEM_PROMPT = (
    "You are a precise, helpful assistant. "
    "Always reason step by step inside <think></think> tags, "
    "then write your final answer inside <answer></answer> tags."
)

# Display configuration overview
print(f"Target HF Dataset Repo : {OUTPUT_DATASET_REPO}")
print(f"Sky-T1 Max Samples     : {MAX_SAMPLES_SKYT1}")
print(f"OpenThoughts Max Samples: {MAX_SAMPLES_OPENTHOUGHT}")

## Cell 3 — Data Cleaning & Format Standardization Logic

In [ ]:
# ==============================================================================
# Cell 3 — Data Cleaning & Format Standardization Functions
# ==============================================================================
import re

def wrap_answer(text):
    """
    Ensures the final answer text is properly enclosed in <answer>...</answer> tags.
    - If already tagged with <answer>, returns as-is.
    - If a LaTeX \boxed{...} expression exists, extracts the inner content as the answer.
    - Otherwise, falls back to wrapping the last 400 characters of text.
    """
    text = text.strip()
    if "<answer>" in text:
        return text
    boxed = re.search(r"\\boxed\{(.+?)\}", text)
    ans = boxed.group(1) if boxed else text[-400:]
    return f"<answer>{ans}</answer>"


# ------------------------------------------------------------------------------
# Sky-T1 Dataset Cleaning Function
# ------------------------------------------------------------------------------
def clean_sky_t1(example):
    """
    Cleans and standardizes raw samples from NovaSky-AI/Sky-T1_data_17k.
    1. Extracts 'user' and 'assistant' conversation turns.
    2. Filters out short non-CoT responses (< 100 characters).
    3. Splits assistant output into reasoning body (<think>) and final answer (<answer>).
    4. Constructs standard OpenAI/ChatML formatted message dictionaries.
    """
    try:
        convs = example.get("conversations", [])
        # Extract user prompt and assistant response from conversation list
        user_msg = next((c["value"] for c in convs if c["from"] == "user"), None)
        asst_msg = next((c["value"] for c in convs if c["from"] == "assistant"), None)
        if not user_msg or not asst_msg:
            return None
        
        asst_msg = asst_msg.strip()
        if len(asst_msg) < 100:   # Filter out trivial responses lacking detailed reasoning
            return None
        
        # Pattern matching to isolate explicit 'Final Answer' declarations
        final_ans_match = re.search(
            r"(?:final answer|therefore,? the answer is|the answer is)[:\s]+(.+?)(?:\n|$)",
            asst_msg, re.IGNORECASE | re.DOTALL
        )
        if final_ans_match:
            ans_text    = final_ans_match.group(1).strip()[:300]
            think_body  = asst_msg[:final_ans_match.start()].strip()
        else:
            # Fallback: Split by double newline and treat the last paragraph as final answer
            paragraphs = [p.strip() for p in asst_msg.split("\n\n") if p.strip()]
            ans_text   = paragraphs[-1][:300] if paragraphs else asst_msg[-300:]
            think_body = "\n\n".join(paragraphs[:-1]) if len(paragraphs) > 1 else asst_msg

        # Wrap final answer in <answer> tags and format assistant completion
        answer_part    = wrap_answer(ans_text)
        assistant_text = f"<think>\n{think_body}\n</think>\n{answer_part}"

        # Return standardized system, user, and assistant message structure
        return {"messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": assistant_text},
        ]}
    except Exception:
        return None


# ------------------------------------------------------------------------------
# OpenThoughts Dataset Cleaning Function
# ------------------------------------------------------------------------------
def clean_openthoughts(example):
    """
    Cleans and standardizes raw samples from open-r1/OpenThoughts-114k-math.
    1. Extracts 'user' and 'assistant' messages from the dataset.
    2. Validates existing <think>...</think> blocks if present.
    3. If missing, separates thought process from final answer automatically.
    4. Returns ChatML structured message objects.
    """
    try:
        msgs     = example.get("messages", [])
        user_msg = next((m["content"] for m in msgs if m["role"] == "user"), None)
        asst_msg = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
        if not user_msg or not asst_msg or len(asst_msg.strip()) < 100:
            return None
            
        asst_msg = asst_msg.strip()
        
        # If <think> tags already exist in OpenThoughts response, preserve and wrap answer part
        if "<think>" in asst_msg and "</think>" in asst_msg:
            think   = re.search(r"<think>(.*?)</think>", asst_msg, re.DOTALL).group(1)
            after   = asst_msg.split("</think>", 1)[-1].strip()
            after   = wrap_answer(after)
            assistant_text = f"<think>{think}</think>\n{after}"
        else:
            # Fallback paragraph splitting if <think> tags are absent
            paragraphs = [p.strip() for p in asst_msg.split("\n\n") if p.strip()]
            ans_text   = paragraphs[-1][:300] if paragraphs else asst_msg[-300:]
            think_body = "\n\n".join(paragraphs[:-1]) if len(paragraphs) > 1 else asst_msg
            assistant_text = f"<think>\n{think_body}\n</think>\n{wrap_answer(ans_text)}"

        return {"messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": assistant_text},
        ]}
    except Exception:
        return None

print("✅ Data cleaning functions successfully defined")

## Cell 4 — Download, Process, Merge & Shuffle Datasets

In [ ]:
# ==============================================================================
# Cell 4 — Dataset Ingestion, Processing, Merging & Shuffling
# ==============================================================================
from datasets import load_dataset, Dataset
import random

# ------------------------------------------------------------------------------
# 1. Download & Process Sky-T1 Dataset (Target: 2,500 samples)
# ------------------------------------------------------------------------------
print("Loading NovaSky-AI/Sky-T1_data_17k...")
sky_raw = load_dataset("NovaSky-AI/Sky-T1_data_17k", split="train").shuffle(seed=SEED)

sky_cleaned = []
for i, ex in enumerate(sky_raw):
    out = clean_sky_t1(ex)
    if out:
        sky_cleaned.append(out)
    if len(sky_cleaned) >= MAX_SAMPLES_SKYT1:
        break
print(f"  Sky-T1 → {len(sky_cleaned)} samples successfully processed")

# ------------------------------------------------------------------------------
# 2. Download & Process OpenThoughts Dataset (Target: 1,500 samples)
# ------------------------------------------------------------------------------
print("Loading open-r1/OpenThoughts-114k-math...")
ot_raw = load_dataset("open-r1/OpenThoughts-114k-math", split="train").shuffle(seed=SEED)

ot_cleaned = []
for i, ex in enumerate(ot_raw):
    out = clean_openthoughts(ex)
    if out:
        ot_cleaned.append(out)
    if len(ot_cleaned) >= MAX_SAMPLES_OPENTHOUGHT:
        break
print(f"  OpenThoughts → {len(ot_cleaned)} samples successfully processed")

# ------------------------------------------------------------------------------
# 3. Combine Datasets, Apply Global Shuffle, and Convert to HuggingFace Dataset
# ------------------------------------------------------------------------------
all_samples = sky_cleaned + ot_cleaned
random.seed(SEED)
random.shuffle(all_samples)

print(f"\n✅ Merged Total: {len(all_samples)} samples")
dataset = Dataset.from_list(all_samples)
print(f"Dataset structure: {dataset}")

## Cell 5 — Push Preprocessed Dataset to Hugging Face Hub

In [ ]:
# ==============================================================================
# Cell 5 — Upload Preprocessed Dataset to Hugging Face Hub
# ==============================================================================

# Push the merged 4,000-sample dataset to the public Hugging Face Dataset Hub
print(f"Pushing dataset to Hugging Face Hub: {OUTPUT_DATASET_REPO} ...")
dataset.push_to_hub(OUTPUT_DATASET_REPO, private=False)

print(f"✅ Preprocessed dataset successfully pushed to:")
print(f"   https://huggingface.co/datasets/{OUTPUT_DATASET_REPO}")